In [113]:
import pandas as pd
import scipy.stats
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
from tqdm import tqdm
from geopy.distance import geodesic

In [94]:
df = pd.read_csv('data/cleaned_card_transactions_2025.csv')

In [95]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90445 entries, 0 to 90444
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Recnum             90445 non-null  int64  
 1   Cardnum            90445 non-null  int64  
 2   Date               90445 non-null  object 
 3   Merchnum           90445 non-null  object 
 4   Merch description  90445 non-null  object 
 5   Merch state        90445 non-null  object 
 6   Merch zip          90445 non-null  object 
 7   Transtype          90445 non-null  object 
 8   Amount             90445 non-null  float64
 9   Fraud              90445 non-null  int64  
 10  amount_okay        90445 non-null  bool   
dtypes: bool(1), float64(1), int64(3), object(6)
memory usage: 7.0+ MB


In [96]:
df['Recnum'] = df['Recnum'].astype(str)
df['Cardnum'] = df['Cardnum'].astype(str)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df = df.drop(columns=['amount_okay', 'Transtype'])

In [97]:
df['DayOfWeek'] = df['Date'].dt.dayofweek

### Entities Creation

In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90445 entries, 0 to 90444
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Recnum             90445 non-null  object        
 1   Cardnum            90445 non-null  object        
 2   Date               90445 non-null  datetime64[ns]
 3   Merchnum           90445 non-null  object        
 4   Merch description  90445 non-null  object        
 5   Merch state        90445 non-null  object        
 6   Merch zip          90445 non-null  object        
 7   Amount             90445 non-null  float64       
 8   Fraud              90445 non-null  int64         
 9   DayOfWeek          90445 non-null  int32         
dtypes: datetime64[ns](1), float64(1), int32(1), int64(1), object(6)
memory usage: 6.6+ MB


In [99]:
# Two-way combinations: Card-based
df['card_merch'] = df['Cardnum'] + '_' + df['Merchnum']
df['card_zip'] = df['Cardnum'] + '_' + df['Merch zip']
df['card_state'] = df['Cardnum'] + '_' + df['Merch state']
df['card_merchdesc'] = df['Cardnum'] + '_' + df['Merch description']
df['card_dow'] = df['Cardnum'] + '_' + df['DayOfWeek'].astype(str)

# Two-way combinations: Merchant-based
df['merch_zip'] = df['Merchnum'] + '_' + df['Merch zip']
df['merch_state'] = df['Merchnum'] + '_' + df['Merch state']
df['merchnum_desc'] = df['Merchnum'] + '_' + df['Merch description']
df['merchnum_dow'] = df['Merchnum'] + '_' + df['DayOfWeek'].astype(str)

# Two-way combinations: Location/Description-based
df['state_desc'] = df['Merch state'] + '_' + df['Merch description']
df['merchdesc_dow'] = df['Merch description'] + '_' + df['DayOfWeek'].astype(str)
df['merchdesc_state'] = df['Merch description'] + '_' + df['Merch state']
df['merchdesc_zip'] = df['Merch description'] + '_' + df['Merch zip']

# Three-way combinations: Card + Merchant + Location/Description
df['card_merchnum_desc'] = df['Cardnum'] + '_' + df['Merchnum'] + '_' + df['Merch description']
df['card_merchnum_zip'] = df['Cardnum'] + '_' + df['Merchnum'] + '_' + df['Merch zip']
df['card_merchnum_state'] = df['Cardnum'] + '_' + df['Merchnum'] + '_' + df['Merch state']
df['card_merchdesc_zip'] = df['Cardnum'] + '_' + df['Merch description'] + '_' + df['Merch zip']
df['card_merchdesc_state'] = df['Cardnum'] + '_' + df['Merch description'] + '_' + df['Merch state']

# Three-way combinations: Merchant + Description + Location
df['merchnum_desc_state'] = df['Merchnum'] + '_' + df['Merch description'] + '_' + df['Merch state']
df['merchnum_desc_zip'] = df['Merchnum'] + '_' + df['Merch description'] + '_' + df['Merch zip']

# Zip3-based combinations (these may increase computation time)
df['zip3'] = df['Merch zip'].str[:3]
df['card_zip3'] = df['Cardnum'] + '_' + df['zip3']
df['state_zip'] = df['Merch state'] + '_' + df['Merch zip']
df['merchnum_zip3'] = df['Merchnum'] + '_' + df['zip3']

We are keeping Cardnum and Merchnum as key grouping entities in the following feature engineering. All others are excluded because they are either embedded in the combination entities or is the subject of aggregation (e.g. Amount).

In [100]:
# entity columns for time-based aggregations
entity_cols = [
    'Cardnum', 'Merchnum',
    'card_merch', 'card_zip', 'card_state', 'card_merchdesc', 'card_dow',
    'merch_zip', 'merch_state', 'merchnum_desc', 'merchnum_dow',
    'state_desc', 'merchdesc_dow', 'merchdesc_state', 'merchdesc_zip',
    'card_merchnum_desc', 'card_merchnum_zip', 'card_merchnum_state',
    'card_merchdesc_zip', 'card_merchdesc_state',
    'merchnum_desc_state', 'merchnum_desc_zip',
    'zip3', 'card_zip3', 'state_zip', 'merchnum_zip3'
]

# Filter out entities with low cardinality (<=10 unique values)
entities_to_keep = []
entities_to_remove = []

for entity in entity_cols:
    unique_count = df[entity].nunique()
    status = "REMOVE" if unique_count <= 10 else "KEEP"
    print(f"{entity:30} {unique_count:>8} unique | {status}")
    
    if unique_count <= 10:
        entities_to_remove.append(entity)
    else:
        entities_to_keep.append(entity)

print(f"\nKept {len(entities_to_keep)} entities, removed {len(entities_to_remove)} low-cardinality entities")

Cardnum                            1642 unique | KEEP
Merchnum                          13605 unique | KEEP
card_merch                        40650 unique | KEEP
card_zip                          34230 unique | KEEP
card_state                        16152 unique | KEEP
card_merchdesc                    41264 unique | KEEP
card_dow                           9170 unique | KEEP
merch_zip                         13796 unique | KEEP
merch_state                       13669 unique | KEEP
merchnum_desc                     14555 unique | KEEP
merchnum_dow                      27920 unique | KEEP
state_desc                        13216 unique | KEEP
merchdesc_dow                     27676 unique | KEEP
merchdesc_state                   13216 unique | KEEP
merchdesc_zip                     13458 unique | KEEP
card_merchnum_desc                43373 unique | KEEP
card_merchnum_zip                 40798 unique | KEEP
card_merchnum_state               40679 unique | KEEP
card_merchdesc_zip          

In [101]:
if entities_to_remove:
    df.drop(columns=entities_to_remove, inplace=True)
    print(f"Dropped {len(entities_to_remove)} columns: {entities_to_remove}")

# Update entity list to only include kept entities
entity_cols = entities_to_keep

print(f"dataframe shape: {df.shape}")
print(f"{len(entity_cols)} entities for feature engineering")

dataframe shape: (90445, 34)
26 entities for feature engineering


### Data Partitioning

In [102]:
df.Date.min(), df.Date.max()

(Timestamp('2010-01-01 00:00:00'), Timestamp('2010-12-31 00:00:00'))

We unfortunately only have a full year of data. Ideally, we would want to have at least one year of data in the training + testing partition to account for seasonality. 

However, given the circumstance, we would have to compromise on this and leave some of the months for real out-of-time validation set for true out-of-sample testing that is realistic in a real-time fraud detection algorithm.

In [103]:
df_oot = df[df.Date >= '2010-11-01']
df_train_test = df[df.Date < '2010-11-01']

### Prepare Dataframes for Time-Based Feature Engineering

In [104]:
# df_main: Main dataframe representing "current" transactions
# df_past: Reference dataframe representing "historical/past" transactions for self-joins
# df_features: Accumulator for all engineered features (starts as copy of df_main)

df_main = df.copy()
df_past = df.copy()
df_features = df.copy()

# Add reference columns to df_past for self-join operations
df_past['past_date'] = df_past['Date']
df_past['past_recnum'] = df_past['Recnum']

### Time-Based Feature Engineering

For each entity, we create:
1. **Days since last transaction**: How many days since this entity was last seen
2. **Transaction frequency**: Count of transactions in various time windows (0, 1, 3, 7, 14, 30, 60 days)
3. **Amount statistics**: Average, max, median, and total amounts per time window
4. **Ratio features**: Current amount relative to historical patterns (actual/avg, actual/max, etc.)

In [105]:
# Time windows for rolling aggregations (in days)
TIME_WINDOWS = [0, 1, 3, 7, 14, 30, 60]

# Collect all new features in a dictionary
new_features = {}

for entity in tqdm(entity_cols, desc="Processing entities"):
    
    # Self-join: current transactions with historical transactions
    merged = pd.merge(
        df_main[['Recnum', 'Date', entity]],
        df_past[['past_recnum', 'past_date', entity, 'Amount']],
        on=entity
    )
    
    # Filter to only past transactions (past_recnum < current Recnum)
    past_txns = merged[merged.Recnum > merged.past_recnum]
    
    # DAYS SINCE LAST TRANSACTION
    last_txn = past_txns.groupby('Recnum')[['Date', 'past_date']].last()
    days_since = (last_txn['Date'] - last_txn['past_date']).dt.days
    
    # For those with no prior transaction, fill with days since 2010-01-01
    new_features[f'{entity}_day_since'] = df_main['Recnum'].map(days_since).fillna(
        (df_main['Date'] - pd.to_datetime('2010-01-01')).dt.days
    )
    
    # ROLLING TIME WINDOW FEATURES
    for days in TIME_WINDOWS:
        # Filter transactions within time window
        window_txns = past_txns[
            past_txns['past_date'] >= (past_txns['Date'] - dt.timedelta(days))
        ][['Recnum', entity, 'Amount']]
        
        # Count of transactions in window
        txn_counts = window_txns.groupby('Recnum')[entity].count()
        new_features[f'{entity}_count_{days}'] = df_main['Recnum'].map(txn_counts)
        
        # Amount aggregations
        amount_agg = window_txns.groupby('Recnum')['Amount'].agg(['mean', 'max', 'median', 'sum'])
        new_features[f'{entity}_avg_{days}'] = df_main['Recnum'].map(amount_agg['mean'])
        new_features[f'{entity}_max_{days}'] = df_main['Recnum'].map(amount_agg['max'])
        new_features[f'{entity}_med_{days}'] = df_main['Recnum'].map(amount_agg['median'])
        new_features[f'{entity}_total_{days}'] = df_main['Recnum'].map(amount_agg['sum'])
        
        # Ratio features: current amount relative to historical patterns
        new_features[f'{entity}_actual/avg_{days}'] = df_main['Amount'] / new_features[f'{entity}_avg_{days}']
        new_features[f'{entity}_actual/max_{days}'] = df_main['Amount'] / new_features[f'{entity}_max_{days}']
        new_features[f'{entity}_actual/med_{days}'] = df_main['Amount'] / new_features[f'{entity}_med_{days}']
        new_features[f'{entity}_actual/total_{days}'] = df_main['Amount'] / new_features[f'{entity}_total_{days}']

# Join all features at once
df_features = pd.concat([df_main, pd.DataFrame(new_features)], axis=1)

Processing entities: 100%|██████████| 26/26 [02:11<00:00,  5.07s/it]



In [106]:
print(f"Original columns: {df.shape[1]}")
print(f"Features after engineering: {df_features.shape[1]}")
print(f"New features created: {df_features.shape[1] - df.shape[1]}")

Original columns: 34
Features after engineering: 1698
New features created: 1664


### Derived Features from Time-Based Aggregations

1. **Velocity ratios**: Compare short-term activity to longer-term patterns
2. **Velocity-to-recency ratios**: Activity rate normalized by days since last transaction  
3. **Amount variability**: Differences between current and past transaction amounts

In [107]:
# VELOCITY RATIOS: Compare short-term vs long-term activity patterns
SHORT_WINDOWS = [0, 1]
LONG_WINDOWS = [7, 14, 30, 60]

for entity in tqdm(entity_cols, desc="Velocity ratios"):
    for short in SHORT_WINDOWS:
        for long in LONG_WINDOWS:
            # Transaction count velocity: short-term count / long-term count / days
            new_features[f'{entity}_count_{short}_by_{long}'] = \
                new_features[f'{entity}_count_{short}'] / new_features[f'{entity}_count_{long}'] / long
            
            # Amount velocity: short-term total / long-term total / days
            new_features[f'{entity}_total_amount_{short}_by_{long}'] = \
                new_features[f'{entity}_total_{short}'] / new_features[f'{entity}_total_{long}'] / long

# VELOCITY-TO-RECENCY RATIOS: Activity rate normalized by days since last transaction
for entity in tqdm(entity_cols, desc="Velocity-to-recency ratios"):
    for short in SHORT_WINDOWS:
        for long in LONG_WINDOWS:
            new_features[f'{entity}_vdratio_{short}by{long}'] = \
                new_features[f'{entity}_count_{short}_by_{long}'] / (new_features[f'{entity}_day_since'] + 1)

# Update df_features with new derived features
df_features = pd.concat([df_features, pd.DataFrame(new_features)], axis=1)

Velocity-to-recency ratios: 100%|██████████| 26/26 [00:00<00:00, 199.84it/s]



In [108]:
# AMOUNT VARIABILITY: Track differences between current and historical transaction amounts
VARIABILITY_WINDOWS = [0, 1, 3, 7, 14, 30]

for entity in tqdm(entity_cols, desc="Amount variability"):
    
    # Self-join with Amount columns
    merged = pd.merge(
        df_main[['Recnum', 'Date', entity, 'Amount']],
        df_past[['past_recnum', 'past_date', entity, 'Amount']],
        on=entity,
        suffixes=('_current', '_past')
    )
    
    # Filter to only past transactions
    past_txns = merged[merged.Recnum > merged.past_recnum]
    
    for days in VARIABILITY_WINDOWS:
        # Filter transactions within time window
        window_txns = past_txns[
            past_txns['past_date'] >= (past_txns['Date'] - dt.timedelta(days))
        ].copy()
        
        # Calculate amount difference (past - current)
        window_txns['amount_diff'] = window_txns['Amount_past'] - window_txns['Amount_current']
        
        # Aggregate variability metrics
        variability_agg = window_txns.groupby('Recnum')['amount_diff'].agg(['mean', 'max', 'median'])
        new_features[f'{entity}_variability_avg_{days}'] = df_main['Recnum'].map(variability_agg['mean'])
        new_features[f'{entity}_variability_max_{days}'] = df_main['Recnum'].map(variability_agg['max'])
        new_features[f'{entity}_variability_med_{days}'] = df_main['Recnum'].map(variability_agg['median'])

# Update df_features with variability features
df_features = pd.concat([df_features, pd.DataFrame(new_features)], axis=1)

Amount variability: 100%|██████████| 26/26 [01:54<00:00,  4.42s/it]



In [109]:
print(f"Total columns: {df_features.shape[1]}")
print(f"Total features created: {df_features.shape[1] - df.shape[1]}")

Total columns: 6742
Total features created: 6708


### Cross-Entity Diversity Features

Track unique counts of one entity within time windows of another entity.
- Example: "How many unique merchants did this card visit in the past 7 days?"
- Captures diversity and breadth of transaction patterns across entities

In [110]:
# CROSS-ENTITY DIVERSITY: Count unique values of one entity within another entity's time windows
DIVERSITY_WINDOWS = [1, 3, 7, 14, 30, 60]

# Iterate through all entity pairs
for entity_i in tqdm(entity_cols, desc="Cross-entity diversity"):
    for entity_j in entity_cols:
        # Skip if same entity
        if entity_i == entity_j:
            continue
        
        # Self-join to get historical transactions
        merged = pd.merge(
            df_main[['Recnum', 'Date', entity_i]],
            df_past[['past_recnum', 'past_date', entity_i, entity_j]],
            on=entity_i
        )
        
        # Filter to only past transactions
        past_txns = merged[merged.Recnum >= merged.past_recnum]
        
        # Calculate unique counts for each time window
        for days in DIVERSITY_WINDOWS:
            window_txns = past_txns[
                past_txns['past_date'] >= (past_txns['Date'] - dt.timedelta(days))
            ]
            
            # Count unique values of entity_j within entity_i's time window
            unique_counts = window_txns.groupby('Recnum')[entity_j].nunique()
            new_features[f'{entity_i}_unique_count_for_{entity_j}_{days}'] = df_main['Recnum'].map(unique_counts)

# Update df_features with diversity features
df_features = pd.concat([df_features, pd.DataFrame(new_features)], axis=1)

Cross-entity diversity:  31%|███       | 8/26 [13:37<30:39, 102.20s/it]  


KeyboardInterrupt: 

In [ ]:
# SQUARED VELOCITY RATIOS: Velocity normalized by squared time windows
# This gives more weight to recent activity relative to longer-term patterns

for entity in tqdm(entity_cols, desc="Squared velocity ratios"):
    for short in SHORT_WINDOWS:
        for long in LONG_WINDOWS:
            new_features[f'{entity}_count_{short}_by_{long}_sq'] = \
                new_features[f'{entity}_count_{short}'] / new_features[f'{entity}_count_{long}'] / (long ** 2)

# Update df_features
df_features = pd.concat([df_features, pd.DataFrame(new_features)], axis=1)

### Additional Business Logic Features

Domain-specific fraud indicators based on transaction patterns and geography.

In [114]:
# GEOGRAPHIC DISTANCE FEATURES
# Calculate distance between consecutive transactions for each card
    
# Load ZIP code database with coordinates
zip_df = pd.read_csv("data/zip_code_database.csv")
zip_df['zip'] = zip_df['zip'].astype(str)

# Create ZIP to coordinates mapping
zip_to_coords = zip_df.set_index('zip')[['latitude', 'longitude']].to_dict('index')

def calculate_distance(zip1, zip2):
    """Calculate distance in miles between two ZIP codes"""
    coords_1 = zip_to_coords.get(str(zip1))
    coords_2 = zip_to_coords.get(str(zip2))
    if coords_1 and coords_2:
        return geodesic(
            (coords_1['latitude'], coords_1['longitude']), 
            (coords_2['latitude'], coords_2['longitude'])
        ).miles
    return None

# Get previous transaction ZIP for each card
df_temp = df_main.copy()
df_temp['prev_merch_zip'] = df_temp.groupby('Cardnum')['Merch zip'].shift(1)

# Calculate distance to last transaction
new_features['distance_to_last_transaction'] = df_temp.apply(
    lambda row: calculate_distance(row['prev_merch_zip'], row['Merch zip']), 
    axis=1
)

# Flag suspicious distances (>1000 miles)
new_features['suspicious_distance_flag'] = (
    new_features['distance_to_last_transaction'] > 1000
).astype(int)

In [ ]:
# AMOUNT CATEGORY BINS
# Discretize amounts into quintiles for categorical analysis

new_features['amount_category'] = pd.qcut(
    df_main['Amount'], 
    q=5, 
    labels=[1, 2, 3, 4, 5]
).astype(int)

print(new_features['amount_category'].value_counts().sort_index())

Amount category distribution:
Amount
1    18090
2    18088
3    18098
4    18083
5    18086
Name: count, dtype: int64


In [116]:
# FOREIGN ZIP CODE FLAG
# Identify transactions at merchants with non-US ZIP codes

us_zips = set(zip_df['zip'].astype(str).values)

new_features['foreign_zip_flag'] = df_main['Merch zip'].apply(
    lambda x: 0 if str(x) in us_zips else 1
)

print(f"Foreign ZIP transactions: {new_features['foreign_zip_flag'].sum()}")

Foreign ZIP transactions: 10297


In [117]:
# WEEKEND TRANSACTION FLAG
# Binary indicator for transactions on Saturday or Sunday

new_features['is_weekend'] = (df_main['DayOfWeek'] >= 5).astype(int)

print(f"Weekend transactions: {new_features['is_weekend'].sum()} ({new_features['is_weekend'].mean()*100:.1f}%)")

Weekend transactions: 22578 (25.0%)


In [118]:
# ROUNDED AMOUNT FLAG
# Flag whole number amounts (potentially suspicious pattern)

new_features['amount_is_rounded'] = (df_main['Amount'] % 1 == 0).astype(int)

print(f"Rounded amounts: {new_features['amount_is_rounded'].sum()} ({new_features['amount_is_rounded'].mean()*100:.1f}%)")

Rounded amounts: 25705 (28.4%)


In [119]:
# HIGH AMOUNT FOR MERCHANT FLAG
# Flag transactions exceeding 2x the merchant's average amount

merchant_avg = df_main.groupby('Merchnum')['Amount'].mean()
df_main['merchant_avg_amount'] = df_main['Merchnum'].map(merchant_avg)

new_features['high_amount_for_merchant'] = (
    df_main['Amount'] > (2 * df_main['merchant_avg_amount'])
).astype(int)

print(f"High amount transactions: {new_features['high_amount_for_merchant'].sum()} ({new_features['high_amount_for_merchant'].mean()*100:.1f}%)")

High amount transactions: 9090 (10.1%)


For each card, what percentage of its transactions go to its single most-visited merchant?

In [ ]:
# MERCHANT DOMINANCE SCORE
# Fraction of card's transactions at its most frequent merchant

card_merchant_counts = df_main.groupby(['Cardnum', 'Merchnum']).size().reset_index(name='txn_count')
most_freq_merchant = card_merchant_counts.loc[
    card_merchant_counts.groupby('Cardnum')['txn_count'].idxmax()
]

total_txns_per_card = df_main.groupby('Cardnum').size().reset_index(name='total_txns')

dominance = pd.merge(most_freq_merchant, total_txns_per_card, on='Cardnum')
dominance['merchant_dominance_score'] = dominance['txn_count'] / dominance['total_txns']

new_features['merchant_dominance_score'] = df_main['Cardnum'].map(
    dominance.set_index('Cardnum')['merchant_dominance_score']
)

print(f"Mean dominance score: {new_features['merchant_dominance_score'].mean():.3f}")

Mean dominance score: 0.199


In [121]:
# STATE INCONSISTENCY FLAG
# Flag if card used in different state within 1 day (impossible travel)

df_sorted = df_main.sort_values(['Cardnum', 'Date'])
df_sorted['prev_state'] = df_sorted.groupby('Cardnum')['Merch state'].shift(1)
df_sorted['prev_date'] = df_sorted.groupby('Cardnum')['Date'].shift(1)
df_sorted['days_since_prev'] = (df_sorted['Date'] - df_sorted['prev_date']).dt.days

new_features['state_inconsistency_flag'] = (
    (df_sorted['Merch state'] != df_sorted['prev_state']) & 
    (df_sorted['days_since_prev'] <= 1)
).fillna(False).astype(int)

print(f"State inconsistencies: {new_features['state_inconsistency_flag'].sum()} ({new_features['state_inconsistency_flag'].mean()*100:.1f}%)")

State inconsistencies: 30114 (33.3%)


In [ ]:
# Update df_features with all business logic features
df_features = pd.concat([df_features, pd.DataFrame(new_features)], axis=1)

print(f"\nFinal feature engineering complete:")
print(f"  Original columns: {df.shape[1]}")
print(f"  Total columns: {df_features.shape[1]}")
print(f"  Features created: {df_features.shape[1] - df.shape[1]}")